In [1]:
# ============================================================================
# KAGGLE SINGLE-CELL NOTEBOOK — Railway YOLOv8s grouped reanalysis
# Auto-resumes across kernel interruptions (idle timeout, GPU quota, crash,
# closed tab) by using a private Kaggle Dataset as persistent checkpoint
# storage. Kaggle has no Google-Drive equivalent, so this replaces that role.
#
# ---------------------------- ONE-TIME SETUP -------------------------------
#   1. Notebook Settings -> Accelerator -> GPU (T4 x2 or P100).
#   2. Notebook Settings -> Internet -> On (required for pip + Kaggle API).
#   3. "+ Add Data" -> attach the "railway-inspection-dataset" Kaggle Dataset
#      (owner: magajiabdulkareem) so it mounts under /kaggle/input. The script
#      checks /kaggle/input/datasets/magajiabdulkareem/railway-inspection-dataset
#      first, then falls back to the standard Kaggle mount path
#      /kaggle/input/railway-inspection-dataset if the first one isn't present.
#   4. Add-ons -> Secrets -> add two secrets:
#        KAGGLE_USERNAME = your Kaggle username
#        KAGGLE_KEY      = your Kaggle API key (kaggle.com/settings -> API)
#      Without these, training still runs, but nothing survives a full
#      kernel restart — resume will NOT work.
#   5. RECOMMENDED FIRST: set EPOCHS = 2 below and run once end-to-end to
#      confirm push/resume actually works for your account before
#      committing GPU-quota hours to the real 150-epoch run.
#   6. Change SEED for seeds 0, 1, 2, 3, 4. Each seed gets its own
#      checkpoint dataset, so seeds don't collide.
#
# To resume after any interruption: just re-open this notebook and run this
# exact same cell again, unchanged. It detects and continues automatically.
#
# ------------------------- OUTPUTS FOR THE PAPER ----------------------------
# Beyond training/eval, this now also produces (all saved under PROJECT_ROOT
# and pushed to your checkpoint dataset):
#   - image_resolution_stats.json      : width/height min/max/mean/std
#   - class_distribution.json          : per-class image & object counts
#   - environment.json                 : now includes CPU model, cores, RAM
#   - metrics_summary.json             : now includes per-class P/R/AP50
#   - evaluations/.../error_analysis/  : annotated FP/FN overlay images +
#                                          error_analysis_summary.json
#   - <base>-aggregate checkpoint      : cross-seed mean/std/95% CI, updated
#                                          automatically as each seed finishes
# ============================================================================

import os, sys, subprocess, json, platform, time, random, re, shutil, hashlib, zipfile
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3.0", "pyyaml>=6.0.2",
                 "opencv-python-headless>=4.10.0.84"], check=True)

import numpy as np
import pandas as pd
import yaml
import torch
from sklearn.model_selection import StratifiedGroupKFold
from ultralytics import YOLO

# ========================== USER SETTINGS ==========================
SEED = 0                  # Run separately for 0, 1, 2, 3, and 4
EPOCHS = 150               # Set to 2 for a dry run first (see setup note above)
IMGSZ = 640
LOCKED_CONFIDENCE = 0.25  # Do NOT tune this using test results.
CHECKPOINT_DATASET_BASE = "railway-yolov8s-grouped-ckpt"  # becomes <username>/<base>-seed{SEED}
# =====================================================================

WORKING = Path("/kaggle/working")
PROJECT_ROOT = WORKING / "railway_yolov8s_grouped_reanalysis"
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = PROJECT_ROOT / "runs" / f"yolov8s_seed{SEED}"
EVAL_DIR = PROJECT_ROOT / "evaluations" / f"yolov8s_seed{SEED}"
COMPLETE_MARKER = PROJECT_ROOT / f"SEED{SEED}_COMPLETE.flag"

assert torch.cuda.is_available(), "Enable GPU: Notebook Settings > Accelerator > GPU."

# --------------------------------------------------------------------
# Kaggle API auth — required for checkpoint persistence across restarts.
# --------------------------------------------------------------------
KAGGLE_AUTH_OK = False
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
    KAGGLE_AUTH_OK = True
except Exception as e:
    print("WARNING: Kaggle API secrets not found/usable — checkpoint persistence across "
          "full kernel restarts will NOT work this run. Add KAGGLE_USERNAME and KAGGLE_KEY "
          "as Secrets (Add-ons > Secrets) to enable auto-resume. Error was:", e)

CHECKPOINT_SLUG = f"{os.environ.get('KAGGLE_USERNAME', 'unknown')}/{CHECKPOINT_DATASET_BASE}-seed{SEED}"

def expand_subdir_zips():
    """Subdirectories are uploaded as single zip files (dir-mode=zip); re-expand them."""
    for zf in PROJECT_ROOT.glob("*.zip"):
        target_dir = PROJECT_ROOT / zf.stem
        if target_dir.exists():
            continue
        try:
            with zipfile.ZipFile(zf) as z:
                z.extractall(target_dir)
            zf.unlink()
            print(f"Restored checkpoint folder: {target_dir.name}/")
        except Exception as e:
            print(f"WARNING: could not expand {zf.name}: {e}")

def checkpoint_pull():
    if not KAGGLE_AUTH_OK:
        return False
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "download", "-d", CHECKPOINT_SLUG,
             "-p", str(PROJECT_ROOT), "--unzip", "--force"],
            capture_output=True, text=True, timeout=1800
        )
        if result.returncode == 0:
            print(f"Found and restored checkpoint dataset: {CHECKPOINT_SLUG}")
            expand_subdir_zips()
            return True
        print("No existing checkpoint dataset found — starting fresh this seed.")
        return False
    except Exception as e:
        print("Checkpoint pull failed — starting fresh:", e)
        return False

def checkpoint_push(message):
    if not KAGGLE_AUTH_OK:
        return
    meta_path = PROJECT_ROOT / "dataset-metadata.json"
    meta_path.write_text(json.dumps({
        "title": f"{CHECKPOINT_DATASET_BASE}-seed{SEED}",
        "id": CHECKPOINT_SLUG,
        "licenses": [{"name": "CC0-1.0"}]
    }, indent=2))
    try:
        create = subprocess.run(
            ["kaggle", "datasets", "create", "-p", str(PROJECT_ROOT), "-r", "zip"],
            capture_output=True, text=True, timeout=1800
        )
        if create.returncode == 0:
            print(f"Checkpoint dataset created: {CHECKPOINT_SLUG}")
            return
        version = subprocess.run(
            ["kaggle", "datasets", "version", "-p", str(PROJECT_ROOT), "-m", message, "-r", "zip"],
            capture_output=True, text=True, timeout=1800
        )
        if version.returncode == 0:
            print(f"Checkpoint pushed: {CHECKPOINT_SLUG} ({message})")
        else:
            print("WARNING: checkpoint push failed:", (version.stderr or "")[:500])
    except Exception as e:
        print("WARNING: checkpoint push failed (training continues locally):", e)

RESUMED_FROM_CHECKPOINT = checkpoint_pull()

if COMPLETE_MARKER.exists():
    raise RuntimeError(
        f"Seed {SEED} already completed (found {COMPLETE_MARKER.name} in restored checkpoint). "
        f"Change SEED to train the next seed; never overwrite a completed run."
    )

# Record environment once, on first start of this seed (preserved across resumes).
env_path = PROJECT_ROOT / "environment.json"
if not env_path.exists():
    try:
        import psutil
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psutil"], check=False)
        import psutil
    cpu_info = {
        "cpu_model": platform.processor() or platform.uname().processor or "unknown",
        "cpu_count_logical": os.cpu_count(),
        "cpu_count_physical": psutil.cpu_count(logical=False),
        "total_ram_gb": round(psutil.virtual_memory().total / (1024 ** 3), 2),
        "available_ram_gb": round(psutil.virtual_memory().available / (1024 ** 3), 2),
    }
    env = {
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "python": sys.version,
        "platform": platform.platform(),
        "cpu": cpu_info,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "gpu_properties": str(torch.cuda.get_device_properties(0)),
        "ultralytics": __import__("ultralytics").__version__,
        "pip_freeze": subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True),
    }
    env_path.write_text(json.dumps(env, indent=2))
    print("CPU:", cpu_info["cpu_model"], "|", cpu_info["cpu_count_logical"], "logical cores |",
          cpu_info["total_ram_gb"], "GB RAM total")
print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------------------------------------------
# Locate the source dataset at the specified path (with a fallback, since
# Kaggle mounts datasets at /kaggle/input/<slug>, not /kaggle/input/datasets/<user>/<slug>).
# --------------------------------------------------------------------
PRIMARY_INPUT_PATH = Path("/kaggle/input/datasets/magajiabdulkareem/railway-inspection-dataset")
FALLBACK_INPUT_PATH = Path("/kaggle/input/railway-inspection-dataset")

if PRIMARY_INPUT_PATH.exists():
    DATASET_INPUT_ROOT = PRIMARY_INPUT_PATH
elif FALLBACK_INPUT_PATH.exists():
    print(f"NOTE: {PRIMARY_INPUT_PATH} not found; using standard Kaggle mount path instead: {FALLBACK_INPUT_PATH}")
    DATASET_INPUT_ROOT = FALLBACK_INPUT_PATH
else:
    available = sorted(p.name for p in Path("/kaggle/input").iterdir()) if Path("/kaggle/input").exists() else []
    raise AssertionError(
        f"Dataset not found at {PRIMARY_INPUT_PATH} or {FALLBACK_INPUT_PATH}. "
        f"Attach the dataset via '+ Add Data' if you haven't. "
        f"Datasets currently attached under /kaggle/input: {available}"
    )
print("Using dataset input:", DATASET_INPUT_ROOT)

# Extract fresh into local scratch every kernel start (cheap; not part of the
# persisted checkpoint since it's fully reproducible from the input dataset + manifest).
RAW_ROOT = Path("/kaggle/working/railway_raw_3381")
if RAW_ROOT.exists():
    shutil.rmtree(RAW_ROOT)
RAW_ROOT.mkdir(parents=True)

zip_candidates = list(DATASET_INPUT_ROOT.rglob("*.zip"))
if zip_candidates:
    print("Found archive inside dataset, extracting:", zip_candidates[0])
    with zipfile.ZipFile(zip_candidates[0]) as z:
        z.extractall(RAW_ROOT)
else:
    # Dataset is already unzipped on Kaggle's side; copy (read-only input can't be used in place
    # for downstream shutil.copy2 source paths reliably across sessions, so mirror it locally).
    print("No zip found inside dataset; treating it as already-extracted YOLO folders.")
    shutil.copytree(DATASET_INPUT_ROOT, RAW_ROOT, dirs_exist_ok=True)

yamls = list(RAW_ROOT.rglob("data.yaml"))
assert len(yamls) == 1, f"Expected one data.yaml; found {yamls}"
SOURCE_ROOT = yamls[0].parent
source_cfg = yaml.safe_load(yamls[0].read_text())
print("Source YAML:", source_cfg)

def normalize_family(filename):
    stem = Path(filename).stem.lower()
    stem = re.sub(r"\.rf\.[0-9a-f]{6,}$", "", stem)
    stem = re.sub(r"^(?:aug[_-]?\d+[_-]?)+", "", stem)
    stem = re.sub(r"^(?:copy[_-]?\d+[_-]?)+", "", stem)
    return stem

def read_label_counts(label_path):
    counts = {0: 0, 1: 0}
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        fields = line.split()
        assert len(fields) == 5, f"Malformed YOLO label: {label_path}"
        cls = int(float(fields[0]))
        assert cls in counts, f"Unexpected class {cls}: {label_path}"
        values = list(map(float, fields[1:]))
        assert all(0 <= x <= 1 for x in values), f"Out-of-range box: {label_path}"
        counts[cls] += 1
    return counts

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

manifest_path = PROJECT_ROOT / "grouped_split_manifest.csv"
if manifest_path.exists():
    # Reuse the exact split restored from checkpoint, for continuity across resumes.
    manifest = pd.read_csv(manifest_path)
    print("Reusing restored split manifest:", len(manifest), "images |",
          manifest.family.nunique(), "families")
else:
    records = []
    for old_split in ["train", "val", "test"]:
        for image in sorted((SOURCE_ROOT / "images" / old_split).iterdir()):
            if image.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
                continue
            label = SOURCE_ROOT / "labels" / old_split / f"{image.stem}.txt"
            assert label.exists(), f"Missing label: {label}"
            c = read_label_counts(label)
            records.append({
                "image_path": str(image), "label_path": str(label), "filename": image.name,
                "original_split": old_split, "family": normalize_family(image.name),
                "defective": c[0], "non_defective": c[1], "objects": c[0] + c[1]
            })

    manifest = pd.DataFrame(records)
    assert len(manifest) > 0, "No images found — check the dataset path and folder structure."
    assert (manifest["objects"] > 0).all(), "Unexpected empty labels; inspect the archive before continuing."
    print(f"Input images: {len(manifest)}  (NOTE: this is 'railway-inspection-dataset' — confirm "
          f"this count matches what you expect before proceeding to training.)")
    print("Apparent families (filename-based):", manifest.family.nunique())

    # Merge families by actual byte content: catches duplicates the filename regex misses.
    manifest["content_hash"] = manifest["image_path"].apply(lambda p: sha256(Path(p)))
    parent = {f: f for f in manifest["family"].unique()}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    merged_groups = 0
    for content_hash, group in manifest.groupby("content_hash"):
        families_here = group["family"].unique()
        if len(families_here) > 1:
            merged_groups += 1
        for f in families_here[1:]:
            union(families_here[0], f)
    manifest["family"] = manifest["family"].apply(find)
    print(f"Content-hash merge: {merged_groups} filename-families merged into shared image families")
    print("Apparent families after content-hash merge:", manifest.family.nunique())

    manifest["stratum"] = np.select(
        [(manifest.defective > 0) & (manifest.non_defective > 0),
         manifest.defective > 0,
         manifest.non_defective > 0],
        ["both", "defective_only", "non_defective_only"],
        default="unknown"
    )
    assert not (manifest["stratum"] == "unknown").any(), "Found images with zero objects in both classes; inspect labels."

    sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=2026)
    fold = np.full(len(manifest), -1)
    for fold_number, (_, test_index) in enumerate(sgkf.split(manifest, manifest.stratum, groups=manifest.family)):
        fold[test_index] = fold_number
    manifest["split"] = np.where(fold == 0, "test", np.where(fold == 1, "val", "train"))
    assert not (manifest.groupby("family").split.nunique() > 1).any(), "Family leakage detected; STOP."

    summary = manifest.groupby("split").agg(
        images=("filename", "count"), families=("family", "nunique"),
        defective_objects=("defective", "sum"),
        non_defective_objects=("non_defective", "sum"), objects=("objects", "sum")
    )
    print("\nNew split summary:\n", summary)
    manifest.to_csv(manifest_path, index=False)
    summary.to_csv(PROJECT_ROOT / "grouped_split_summary.csv")
    (PROJECT_ROOT / "split_protocol.txt").write_text(
        "Secondary-data reanalysis. Filename-family grouping removed Roboflow hash suffixes and aug_N_ prefixes, "
        "then families were further merged by exact byte-content hash to catch duplicates filenames alone missed. "
        "StratifiedGroupKFold(10 folds, random_state=2026): fold 0=test, fold 1=validation, folds 2-9=training. "
        "This is content-duplicate-disjoint and augmentation-family-disjoint, not proven raw-source-disjoint.\n"
    )
    checkpoint_push("Initial split manifest")

# --------------------------------------------------------------------
# Image resolution statistics (reviewer comment #1) — computed once,
# cached across resumes.
# --------------------------------------------------------------------
resolution_stats_path = PROJECT_ROOT / "image_resolution_stats.json"
if not resolution_stats_path.exists():
    from PIL import Image
    widths, heights = [], []
    unreadable = []
    for image_path in manifest["image_path"]:
        try:
            with Image.open(image_path) as im:
                w, h = im.size
            widths.append(w); heights.append(h)
        except Exception as e:
            unreadable.append(str(image_path))
    widths_arr, heights_arr = np.array(widths), np.array(heights)
    resolution_stats = {
        "n_images_measured": int(len(widths_arr)),
        "n_unreadable": len(unreadable),
        "width_px": {"min": int(widths_arr.min()), "max": int(widths_arr.max()),
                     "mean": round(float(widths_arr.mean()), 2), "std": round(float(widths_arr.std()), 2)},
        "height_px": {"min": int(heights_arr.min()), "max": int(heights_arr.max()),
                      "mean": round(float(heights_arr.mean()), 2), "std": round(float(heights_arr.std()), 2)},
        "unique_resolutions": int(len(set(zip(widths_arr.tolist(), heights_arr.tolist())))),
    }
    if unreadable:
        resolution_stats["unreadable_sample"] = unreadable[:10]
    resolution_stats_path.write_text(json.dumps(resolution_stats, indent=2))
    print("\nIMAGE RESOLUTION STATISTICS:\n", json.dumps(resolution_stats, indent=2))
else:
    print("Resolution stats already computed (restored from checkpoint); skipping recomputation.")

# --------------------------------------------------------------------
# Per-class image and object counts (reviewer comment #7).
# --------------------------------------------------------------------
class_distribution_path = PROJECT_ROOT / "class_distribution.json"
if not class_distribution_path.exists():
    class_distribution = {
        "images_with_any_defective_object": int((manifest["defective"] > 0).sum()),
        "images_with_any_non_defective_object": int((manifest["non_defective"] > 0).sum()),
        "images_with_both_classes": int(((manifest["defective"] > 0) & (manifest["non_defective"] > 0)).sum()),
        "total_defective_objects": int(manifest["defective"].sum()),
        "total_non_defective_objects": int(manifest["non_defective"].sum()),
        "total_objects": int(manifest["objects"].sum()),
        "per_split": json.loads(manifest.groupby("split").agg(
            images=("filename", "count"),
            defective_objects=("defective", "sum"),
            non_defective_objects=("non_defective", "sum")
        ).to_json(orient="index"))
    }
    class_distribution_path.write_text(json.dumps(class_distribution, indent=2))
    print("\nCLASS DISTRIBUTION:\n", json.dumps(class_distribution, indent=2))

# Materialize the grouped YOLO folders locally (always rebuilt fresh; not part of checkpoint).
# Images are SYMLINKED rather than copied — copying would duplicate the entire dataset's
# storage footprint on disk. Labels are tiny text files, so those are copied normally.
GROUPED_ROOT = Path("/kaggle/working/railway_grouped_yolo")
if GROUPED_ROOT.exists():
    shutil.rmtree(GROUPED_ROOT)
for split in ["train", "val", "test"]:
    (GROUPED_ROOT / "images" / split).mkdir(parents=True)
    (GROUPED_ROOT / "labels" / split).mkdir(parents=True)
for row in manifest.itertuples(index=False):
    dest_image = GROUPED_ROOT / "images" / row.split / row.filename
    os.symlink(Path(row.image_path).resolve(), dest_image)
    shutil.copy2(row.label_path, GROUPED_ROOT / "labels" / row.split / f"{Path(row.filename).stem}.txt")

DATA_YAML = GROUPED_ROOT / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(GROUPED_ROOT), "train": "images/train", "val": "images/val", "test": "images/test",
    "names": {0: "defective", 1: "non-defective"}
}, sort_keys=False))

# Audit exact-byte duplicates: they must not cross the new partitions.
seen = {}
for split in ["train", "val", "test"]:
    for p in (GROUPED_ROOT / "images" / split).iterdir():
        seen.setdefault(sha256(p), set()).add(split)
cross_split_exact = {h: s for h, s in seen.items() if len(s) > 1}
assert not cross_split_exact, f"Exact duplicates remain across splits: {len(cross_split_exact)}"
print("Exact-byte duplicate audit: PASSED")

# --------------------------------------------------------------------
# Train. Auto-resumes if a restored last.pt is present for this seed.
# --------------------------------------------------------------------
if (RUN_DIR / "weights" / "best.pt").exists():
    raise RuntimeError(f"Seed {SEED} appears already complete (best.pt exists). Change SEED.")

RESUME_TRAINING = (RUN_DIR / "weights" / "last.pt").exists()

train_args = dict(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=-1, optimizer="AdamW",
    seed=SEED, deterministic=True, workers=2, device=0, pretrained=True, patience=50,
    project=str(PROJECT_ROOT / "runs"), name=f"yolov8s_seed{SEED}", exist_ok=True,
    # save_period=-1: only last.pt/best.pt are kept (no extra epochN.pt snapshots eating disk).
    # cache=False: Ultralytics won't build its own full resized-image disk/RAM cache, which
    # would otherwise duplicate a large chunk of the dataset's footprint again. This trades
    # some training speed for disk safety on a tight quota. If you have >=8GB free system RAM
    # (not disk) and want the speed back, cache="ram" is a safer alternative than "disk" here.
    save=True, save_period=-1, cache=False, plots=True,
    fliplr=0.5, flipud=0.0, degrees=0.0, translate=0.05, scale=0.10, shear=0.0,
    mosaic=0.50, close_mosaic=10, mixup=0.0, copy_paste=0.0
)
if not RESUME_TRAINING:
    (PROJECT_ROOT / f"train_args_seed{SEED}.yaml").write_text(yaml.safe_dump(train_args, sort_keys=False))

CHECKPOINT_PUSH_EVERY_N_EPOCHS = 10  # throttle remote pushes; local last.pt still updates every epoch

def push_checkpoint_callback(trainer):
    epoch_num = trainer.epoch + 1
    if epoch_num % CHECKPOINT_PUSH_EVERY_N_EPOCHS == 0 or epoch_num == trainer.epochs:
        checkpoint_push(f"Checkpoint at epoch {epoch_num}")

if RESUME_TRAINING:
    print(f"Resuming interrupted training for seed {SEED} from last.pt")
    model = YOLO(str(RUN_DIR / "weights" / "last.pt"))
    model.add_callback("on_model_save", push_checkpoint_callback)
    model.train(resume=True)
else:
    model = YOLO("yolov8s.pt")
    model.add_callback("on_model_save", push_checkpoint_callback)
    model.train(**train_args)

BEST = RUN_DIR / "weights" / "best.pt"
assert BEST.exists(), "best.pt was not produced. Inspect the saved training run before continuing."
checkpoint_push("Training complete, before evaluation")

# --------------------------------------------------------------------
# Frozen held-out test evaluation. conf=0.001 is only for AP over the score curve.
# --------------------------------------------------------------------
model = YOLO(str(BEST))
metrics = model.val(
    data=str(DATA_YAML), split="test", imgsz=IMGSZ, conf=0.001, iou=0.7, device=0,
    project=str(PROJECT_ROOT / "evaluations"), name=f"yolov8s_seed{SEED}",
    exist_ok=True, save_json=True, plots=True, verbose=True
)

SAVE_LOCKED_PREDICTION_IMAGES = False  # keep False on tight disk budgets; error_analysis/ already
                                        # provides annotated FP/FN samples. Set True to also save
                                        # full annotated images for every test image (adds real disk use).
model.predict(
    source=str(GROUPED_ROOT / "images" / "test"), imgsz=IMGSZ, conf=LOCKED_CONFIDENCE,
    iou=0.7, device=0, save=SAVE_LOCKED_PREDICTION_IMAGES, save_txt=True, save_conf=True,
    project=str(PROJECT_ROOT / "predictions"), name=f"yolov8s_seed{SEED}_conf{LOCKED_CONFIDENCE}",
    exist_ok=True, verbose=False
)

box = metrics.box
CLASS_NAMES_LOOKUP = {0: "defective", 1: "non-defective"}
class_indices = [int(x) for x in box.ap_class_index]
# NOTE: indexing arrays directly by class id (not by position within ap_class_index)
# mirrors the convention already used for per_class_mAP50_95 below (box.maps[cls_idx]).
# If your Ultralytics version indexes these positionally instead, this will raise an
# IndexError rather than silently producing wrong numbers — that's a version-check signal.
per_class_metrics = {}
for cls_idx in class_indices:
    per_class_metrics[CLASS_NAMES_LOOKUP.get(cls_idx, str(cls_idx))] = {
        "precision": float(box.p[cls_idx]),
        "recall": float(box.r[cls_idx]),
        "AP50": float(box.ap50[cls_idx]),
        "AP50_95": float(box.maps[cls_idx]),
    }
result = {
    "seed": SEED, "checkpoint": str(BEST), "evaluation_split": "frozen grouped test set",
    "precision": float(box.mp), "recall": float(box.mr),
    "mAP50": float(box.map50), "mAP50_95": float(box.map),
    "class_indices": class_indices,
    "per_class_mAP50_95": [float(x) for x in box.maps[box.ap_class_index]],
    "per_class_metrics": per_class_metrics
}
EVAL_DIR.mkdir(parents=True, exist_ok=True)
(EVAL_DIR / "metrics_summary.json").write_text(json.dumps(result, indent=2))
# Also drop a top-level copy: since dir-mode=zip only zips SUBdirectories, this small
# file stays individually downloadable via `kaggle datasets download -f`, so cross-seed
# aggregation below doesn't need to pull each other seed's full weights archive just
# to read one small JSON.
(PROJECT_ROOT / f"metrics_summary_seed{SEED}.json").write_text(json.dumps(result, indent=2))
print("\nFROZEN TEST RESULTS:\n", json.dumps(result, indent=2))

# --------------------------------------------------------------------
# Timing: end-to-end YOLO model-call time only.
# --------------------------------------------------------------------
test_images = sorted((GROUPED_ROOT / "images" / "test").glob("*"))
sample = random.Random(2026).sample(test_images, min(200, len(test_images)))
for p in sample[:20]:
    model.predict(str(p), imgsz=IMGSZ, conf=LOCKED_CONFIDENCE, device=0, verbose=False)
times_ms = []
for p in sample:
    torch.cuda.synchronize(); t0 = time.perf_counter()
    model.predict(str(p), imgsz=IMGSZ, conf=LOCKED_CONFIDENCE, device=0, verbose=False)
    torch.cuda.synchronize(); times_ms.append((time.perf_counter() - t0) * 1000)
latency = {
    "seed": SEED, "n_test_images": len(sample), "warmup_images": 20,
    "median_ms_per_image": float(np.median(times_ms)), "mean_ms_per_image": float(np.mean(times_ms)),
    "p25_ms_per_image": float(np.percentile(times_ms, 25)), "p75_ms_per_image": float(np.percentile(times_ms, 75)),
    "definition": "YOLO model-call time: preprocessing, inference, and postprocessing; excludes capture, transfer, alerts, and human review."
}
(EVAL_DIR / "latency.json").write_text(json.dumps(latency, indent=2))
pd.DataFrame({"image": [p.name for p in sample], "milliseconds": times_ms}).to_csv(EVAL_DIR / "latency_per_image.csv", index=False)
print("\nLATENCY:\n", json.dumps(latency, indent=2))

# --------------------------------------------------------------------
# Error analysis (reviewer comment #13): identify false positives and
# false negatives on the frozen test set at the locked confidence
# threshold, with a sample of annotated overlay images for manual review.
# --------------------------------------------------------------------
import cv2

def yolo_txt_to_boxes(txt_path, img_w, img_h, has_conf=False):
    boxes = []
    if not txt_path.exists():
        return boxes
    for line in txt_path.read_text().splitlines():
        if not line.strip():
            continue
        parts = line.split()
        cls = int(float(parts[0]))
        cx, cy, w, h = map(float, parts[1:5])
        conf = float(parts[5]) if has_conf and len(parts) > 5 else None
        x1, y1 = (cx - w / 2) * img_w, (cy - h / 2) * img_h
        x2, y2 = (cx + w / 2) * img_w, (cy + h / 2) * img_h
        boxes.append({"cls": cls, "box": (x1, y1, x2, y2), "conf": conf})
    return boxes

def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

PRED_LABELS_DIR = PROJECT_ROOT / "predictions" / f"yolov8s_seed{SEED}_conf{LOCKED_CONFIDENCE}" / "labels"
TEST_LABELS_DIR = GROUPED_ROOT / "labels" / "test"
ERROR_DIR = EVAL_DIR / "error_analysis"
ERROR_DIR.mkdir(parents=True, exist_ok=True)
IOU_MATCH_THRESHOLD = 0.5
MAX_OVERLAY_IMAGES = 40

error_rows = []
overlays_saved = 0
for image_path in sorted((GROUPED_ROOT / "images" / "test").iterdir()):
    img = cv2.imread(str(image_path))
    if img is None:
        continue
    h, w = img.shape[:2]
    gt_boxes = yolo_txt_to_boxes(TEST_LABELS_DIR / f"{image_path.stem}.txt", w, h)
    pred_boxes = yolo_txt_to_boxes(PRED_LABELS_DIR / f"{image_path.stem}.txt", w, h, has_conf=True)

    matched_gt_idx, matched_pred_idx = set(), set()
    for pi, pred in enumerate(pred_boxes):
        best_iou, best_gi = 0.0, -1
        for gi, gt in enumerate(gt_boxes):
            if gi in matched_gt_idx or gt["cls"] != pred["cls"]:
                continue
            current_iou = box_iou(pred["box"], gt["box"])
            if current_iou > best_iou:
                best_iou, best_gi = current_iou, gi
        if best_iou >= IOU_MATCH_THRESHOLD:
            matched_gt_idx.add(best_gi)
            matched_pred_idx.add(pi)

    fn_idx = [gi for gi in range(len(gt_boxes)) if gi not in matched_gt_idx]
    fp_idx = [pi for pi in range(len(pred_boxes)) if pi not in matched_pred_idx]

    if fn_idx or fp_idx:
        error_rows.append({
            "image": image_path.name,
            "n_ground_truth": len(gt_boxes), "n_predicted": len(pred_boxes),
            "n_false_negative": len(fn_idx), "n_false_positive": len(fp_idx)
        })
        if overlays_saved < MAX_OVERLAY_IMAGES:
            overlay = img.copy()
            for gi, gt in enumerate(gt_boxes):
                x1, y1, x2, y2 = map(int, gt["box"])
                color = (0, 0, 255) if gi in fn_idx else (0, 200, 0)  # red=missed GT, green=matched GT
                cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
                cv2.putText(overlay, f"GT:{CLASS_NAMES_LOOKUP.get(gt['cls'], gt['cls'])}",
                            (x1, max(0, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            for pi, pred in enumerate(pred_boxes):
                x1, y1, x2, y2 = map(int, pred["box"])
                color = (255, 0, 0) if pi in fp_idx else (0, 200, 200)  # blue=extra pred, yellow=matched pred
                cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 1)
                label = f"P:{CLASS_NAMES_LOOKUP.get(pred['cls'], pred['cls'])}"
                if pred["conf"] is not None:
                    label += f" {pred['conf']:.2f}"
                cv2.putText(overlay, label, (x1, min(h - 1, y2 + 12)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            cv2.imwrite(str(ERROR_DIR / image_path.name), overlay)
            overlays_saved += 1

error_df = pd.DataFrame(error_rows)
if len(error_df):
    error_df.to_csv(EVAL_DIR / "error_analysis_detail.csv", index=False)
error_summary = {
    "seed": SEED,
    "locked_confidence": LOCKED_CONFIDENCE,
    "iou_match_threshold": IOU_MATCH_THRESHOLD,
    "n_test_images": len(list((GROUPED_ROOT / "images" / "test").iterdir())),
    "images_with_any_error": len(error_df),
    "total_false_negatives": int(error_df["n_false_negative"].sum()) if len(error_df) else 0,
    "total_false_positives": int(error_df["n_false_positive"].sum()) if len(error_df) else 0,
    "overlay_images_saved": overlays_saved,
    "overlay_legend": "GT boxes: red=missed (false negative), green=matched. "
                       "Predicted boxes: blue=extra (false positive), yellow=matched.",
}
(EVAL_DIR / "error_analysis_summary.json").write_text(json.dumps(error_summary, indent=2))
print("\nERROR ANALYSIS:\n", json.dumps(error_summary, indent=2))

# Free scratch disk: nothing after this point needs the raw extraction or the
# symlinked grouped-split folders (GROUPED_ROOT itself is mostly symlinks, so
# this mainly reclaims RAW_ROOT — the one real copy of the dataset on disk).
for scratch_dir in (RAW_ROOT, GROUPED_ROOT):
    try:
        shutil.rmtree(scratch_dir, ignore_errors=True)
    except Exception:
        pass
print("Freed scratch dataset folders from local disk.")

COMPLETE_MARKER.write_text(f"Seed {SEED} completed at {time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}\n")
checkpoint_push(f"Seed {SEED} COMPLETE")

# --------------------------------------------------------------------
# Cross-seed aggregation with confidence intervals (reviewer comment #11).
# Best-effort: pulls each OTHER seed's private checkpoint dataset (if it
# exists) just to read its metrics_summary.json, then reports mean/std/95%
# CI across however many of the 5 seeds have finished so far. Re-running
# this after each seed completes keeps the aggregate current.
# --------------------------------------------------------------------
ALL_SEEDS = [0, 1, 2, 3, 4]

def pull_seed_metrics_summary(target_seed):
    if not KAGGLE_AUTH_OK:
        return None
    slug = f"{os.environ.get('KAGGLE_USERNAME', 'unknown')}/{CHECKPOINT_DATASET_BASE}-seed{target_seed}"
    tmp_dir = Path(f"/kaggle/working/_peek_seed{target_seed}")
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True)
    try:
        # Lightweight: fetch just the one small top-level JSON, not the whole checkpoint
        # dataset (which would otherwise pull down that seed's full model weights too).
        filename = f"metrics_summary_seed{target_seed}.json"
        result = subprocess.run(
            ["kaggle", "datasets", "download", "-d", slug, "-f", filename,
             "-p", str(tmp_dir), "--force"],
            capture_output=True, text=True, timeout=120
        )
        if result.returncode != 0:
            return None
        matches = list(tmp_dir.rglob("*.json")) + list(tmp_dir.rglob("*.zip"))
        for m in matches:
            if m.suffix == ".zip":
                with zipfile.ZipFile(m) as z:
                    z.extractall(tmp_dir)
        json_matches = list(tmp_dir.rglob(filename))
        return json.loads(json_matches[0].read_text()) if json_matches else None
    except Exception:
        return None
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

if KAGGLE_AUTH_OK:
    per_seed_results = {SEED: result}
    for other_seed in ALL_SEEDS:
        if other_seed == SEED:
            continue
        fetched = pull_seed_metrics_summary(other_seed)
        if fetched:
            per_seed_results[other_seed] = fetched

    completed_seeds = sorted(per_seed_results.keys())
    print(f"\nCross-seed aggregation: {len(completed_seeds)}/{len(ALL_SEEDS)} seeds available ({completed_seeds})")

    if len(completed_seeds) >= 2:
        import math
        try:
            from scipy import stats as scipy_stats
            HAVE_SCIPY = True
        except ImportError:
            HAVE_SCIPY = False

        def mean_std_ci(values):
            n = len(values)
            m = float(np.mean(values))
            s = float(np.std(values, ddof=1)) if n > 1 else 0.0
            if n > 1:
                t_crit = float(scipy_stats.t.ppf(0.975, df=n - 1)) if HAVE_SCIPY else 1.96
            else:
                t_crit = 0.0
            half_width = t_crit * s / math.sqrt(n) if n > 1 else 0.0
            return {"mean": m, "std": s, "n": n, "ci95_low": m - half_width, "ci95_high": m + half_width,
                    "ci_method": "t-distribution" if HAVE_SCIPY else "normal approximation (scipy unavailable)"}

        cross_seed_summary = {"seeds_included": completed_seeds, "n_seeds_expected": len(ALL_SEEDS)}
        for metric_name in ["precision", "recall", "mAP50", "mAP50_95"]:
            values = [per_seed_results[s][metric_name] for s in completed_seeds]
            cross_seed_summary[metric_name] = mean_std_ci(values)
        cross_seed_summary["per_seed_raw"] = {str(s): per_seed_results[s] for s in completed_seeds}

        print("\nCROSS-SEED SUMMARY:\n", json.dumps(cross_seed_summary, indent=2))

        AGG_DIR = Path("/kaggle/working/railway_yolov8s_grouped_reanalysis_aggregate")
        AGG_DIR.mkdir(parents=True, exist_ok=True)
        (AGG_DIR / "cross_seed_summary.json").write_text(json.dumps(cross_seed_summary, indent=2))
        agg_slug = f"{os.environ.get('KAGGLE_USERNAME', 'unknown')}/{CHECKPOINT_DATASET_BASE}-aggregate"
        (AGG_DIR / "dataset-metadata.json").write_text(json.dumps({
            "title": f"{CHECKPOINT_DATASET_BASE}-aggregate", "id": agg_slug,
            "licenses": [{"name": "CC0-1.0"}]
        }, indent=2))
        try:
            create = subprocess.run(["kaggle", "datasets", "create", "-p", str(AGG_DIR), "-r", "zip"],
                                     capture_output=True, text=True, timeout=600)
            if create.returncode != 0:
                subprocess.run(["kaggle", "datasets", "version", "-p", str(AGG_DIR),
                                 "-m", f"Updated after seed {SEED}", "-r", "zip"],
                                capture_output=True, text=True, timeout=600)
            print(f"Cross-seed summary pushed: {agg_slug}")
        except Exception as e:
            print("WARNING: could not push cross-seed summary:", e)
    else:
        print("Fewer than 2 seeds completed so far — confidence intervals need at least 2 to compute. "
              "Run the remaining seeds and this aggregation will pick them up automatically.")
else:
    print("Kaggle API secrets not configured — cannot pull other seeds' results for cross-seed aggregation.")

print(f"\nCOMPLETE: seed {SEED}. Change SEED and rerun this entire cell for the remaining seeds.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 4.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
CPU: x86_64 | 4 logical cores | 31.35 GB RAM total
GPU: Tesla T4
Using dataset input: /kaggle/input/datasets/magajiabdulkareem/railway-inspection-dataset
No zip found inside dataset; treating it as already-extracted YOLO folders.
Source YAML: {'train': './images/train', 'val': './images/val', 'test': './images

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/150      4.74G      1.092      1.439      1.737         11        640: 100% ━━━━━━━━━━━━ 853/853 4.5it/s 3:11
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 52/52 3.8it/s 13.7s
                   all       1764       1802      0.397      0.678      0.509       0.31

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/150      4.75G      1.025      1.346       1.68         17        640: 100% ━━━━━━━━━━━━ 853/853 4.5it/s 3:10
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 52/52 3.8it/s 13.7s
                   all       1764       1802      0.619      0.671      0.627      0.387

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/150      4.75G      0.962      1.263      1.628         10        640: 100% ━━━━━━━━━━━━ 853/853 4.5it/s 3:09
                 Class     Images  Instances      Box(